# Complaints to Neo4j Desktop (Local)

Este notebook carga **Complaints** a tu instancia local de Neo4j Desktop.

## ⚠️ Requisitos:

1. **Neo4j Desktop** ejecutándose
2. **Recalls e Investigations ya subidos** (notebooks 4.1.1 y 4.2.1)
3. **Archivo CSV** ya generado: `data/neo4j/exports/complaints_neo4j_ready.csv`

In [1]:
import pandas as pd
from neo4j import GraphDatabase
from pathlib import Path

print("="*70)
print("CONFIGURACION NEO4J DESKTOP LOCAL")
print("="*70)

# Credenciales de Neo4j Desktop (LOCAL)
NEO4J_URI  = "bolt://localhost:7687"  # Local
NEO4J_USER = "neo4j"
NEO4J_PASS = "proyectotec"  # Cambiar a tu password real

# Inicializar driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# Verificar conexión
try:
    driver.verify_connectivity()
    print(f"[OK] Conectado a Neo4j Desktop en {NEO4J_URI}")
except Exception as e:
    print(f"[ERROR] No se puede conectar: {e}")
    print("[!] Asegurate de que Neo4j Desktop esté corriendo")

CONFIGURACION NEO4J DESKTOP LOCAL
[OK] Conectado a Neo4j Desktop en bolt://localhost:7687


## Verificar Recalls e Investigations Existentes

In [2]:
with driver.session(database="neo4j") as s:
    recalls = s.run("MATCH (r:Recall) RETURN count(r) AS n").single()['n']
    investigations = s.run("MATCH (i:Investigation) RETURN count(i) AS n").single()['n']
    
    if recalls == 0:
        print("[!] No hay Recalls en Neo4j. Ejecuta primero el notebook 4.1.1")
    else:
        print(f"[OK] Recalls existentes: {recalls:,}")
    
    if investigations == 0:
        print("[!] No hay Investigations en Neo4j. Ejecuta primero el notebook 4.2.1")
    else:
        print(f"[OK] Investigations existentes: {investigations:,}")

[OK] Recalls existentes: 12,760
[OK] Investigations existentes: 4,031


## Cypher para Subir Complaints

In [3]:
CYPHER_UPSERT_COMPLAINT = """
UNWIND $rows AS row

MERGE (c:Complaint {id: toString(row.complaint_id)})
  SET c.complaint_id = toString(row.complaint_id),
      c.make         = row.make,
      c.model        = row.model,
      c.year         = CASE WHEN row.year IS NULL OR row.year = '' THEN NULL ELSE toInteger(row.year) END,
      c.component    = row.component,
      c.description  = coalesce(row.description, ''),
      c.open_date    = coalesce(row.open_date, ''),
      c.fail_date    = coalesce(row.fail_date, ''),
      c.miles        = CASE WHEN row.miles = '' THEN NULL ELSE coalesce(row.miles, '') END,
      c.city         = coalesce(row.city, ''),
      c.state        = coalesce(row.state, ''),
      c.crash        = coalesce(row.crash, ''),
      c.fire         = coalesce(row.fire, ''),
      c.injured      = coalesce(row.injured, ''),
      c.deaths       = coalesce(row.deaths, '')

FOREACH (_ IN CASE WHEN row.make <> '' AND row.make IS NOT NULL THEN [1] ELSE [] END |
  MERGE (mk:Make {name: row.make})
  MERGE (c)-[:OF_MAKE]->(mk)
)

FOREACH (_ IN CASE WHEN row.model <> '' AND row.model IS NOT NULL AND row.make <> '' AND row.make IS NOT NULL THEN [1] ELSE [] END |
  MERGE (md:Model {name: row.model, make: row.make})
  MERGE (c)-[:OF_MODEL]->(md)
)

MERGE (comp:Component {name: coalesce(row.comp_l1, row.component)})
  ON CREATE SET comp.name_lower = toLower(coalesce(row.comp_l1, row.component))
  ON MATCH  SET comp.name_lower = coalesce(comp.name_lower, toLower(coalesce(row.comp_l1, row.component)))
MERGE (c)-[:ABOUT]->(comp)

RETURN count(c) AS upserted;
"""


## Cargar CSV y Subir

In [4]:
CSV = Path("../data/neo4j/exports/complaints_neo4j_ready.csv")
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
print(f"[i] CSV cargado: {len(df):,} filas")
print(f"[i] Columnas: {list(df.columns)}")

[i] CSV cargado: 512,725 filas
[i] Columnas: ['complaint_id', 'make', 'model', 'year', 'component', 'description', 'open_date', 'fail_date', 'miles', 'city', 'state', 'crash', 'fire', 'injured', 'deaths', 'comp_l1']


## Limpiar Relaciones RELATES_TO Incorrectas (Opcional)


In [15]:
# Celda para limpiar relaciones ABOUT en lotes (para evitar OutOfMemoryError)
# Neo4j Desktop tiene memoria limitada, así que eliminamos en batches

CLEANUP_ABOUT_BATCH = """
MATCH (c:Complaint)-[r:ABOUT]->(comp:Component)
WITH r
LIMIT $batch_size
DELETE r
RETURN count(r) AS deleted
"""

def delete_about_batch(batch_size=5000):
    """Eliminar relaciones ABOUT en lotes para evitar OutOfMemoryError"""
    total_deleted = 0
    
    with driver.session(database="neo4j") as s:
        while True:
            result = s.run(CLEANUP_ABOUT_BATCH, batch_size=batch_size).single()
            deleted = result['deleted'] if result else 0
            total_deleted += deleted
            print(f"→ Eliminadas: {deleted:,} (Total: {total_deleted:,})")
            
            if deleted == 0:
                break
    
    print(f"\n[OK] Total relaciones ABOUT eliminadas: {total_deleted:,}")
    return total_deleted

# Ejecutar limpieza por lotes
print("="*70)
print("ELIMINANDO RELACIONES ABOUT EN LOTES")
print("="*70)
delete_about_batch(batch_size=5000)


ELIMINANDO RELACIONES ABOUT EN LOTES
→ Eliminadas: 5,000 (Total: 5,000)
→ Eliminadas: 5,000 (Total: 10,000)
→ Eliminadas: 5,000 (Total: 15,000)
→ Eliminadas: 5,000 (Total: 20,000)
→ Eliminadas: 5,000 (Total: 25,000)
→ Eliminadas: 5,000 (Total: 30,000)
→ Eliminadas: 5,000 (Total: 35,000)
→ Eliminadas: 5,000 (Total: 40,000)
→ Eliminadas: 5,000 (Total: 45,000)
→ Eliminadas: 5,000 (Total: 50,000)
→ Eliminadas: 5,000 (Total: 55,000)
→ Eliminadas: 5,000 (Total: 60,000)
→ Eliminadas: 5,000 (Total: 65,000)
→ Eliminadas: 5,000 (Total: 70,000)
→ Eliminadas: 5,000 (Total: 75,000)
→ Eliminadas: 5,000 (Total: 80,000)
→ Eliminadas: 5,000 (Total: 85,000)
→ Eliminadas: 5,000 (Total: 90,000)
→ Eliminadas: 5,000 (Total: 95,000)
→ Eliminadas: 5,000 (Total: 100,000)
→ Eliminadas: 5,000 (Total: 105,000)
→ Eliminadas: 5,000 (Total: 110,000)
→ Eliminadas: 5,000 (Total: 115,000)
→ Eliminadas: 5,000 (Total: 120,000)
→ Eliminadas: 5,000 (Total: 125,000)
→ Eliminadas: 5,000 (Total: 130,000)
→ Eliminadas: 5,000 (

512725

In [13]:
# Si ya subiste Complaints con las relaciones RELATES_TO incorrectas, ejecuta esta celda para eliminarlas
CLEANUP_CYPHER = """
MATCH (c:Complaint)-[r:ABOUT]->(rec:comp)
DELETE r
RETURN count(r) AS deleted_rels
"""

with driver.session(database="neo4j") as s:
    result = s.run(CLEANUP_CYPHER).single()
    deleted = result['deleted_rels'] if result else 0
    print(f"[OK] Relaciones ABOUT eliminadas: {deleted:,}")


[OK] Relaciones ABOUT eliminadas: 0


In [5]:
def ingest(csv_df, cypher, batch=1000):
    total, i = len(csv_df), 0
    with driver.session(database="neo4j") as s:
        while i < total:
            rows = csv_df.iloc[i:i+batch].to_dict('records')
            s.run(cypher, rows=rows)
            i += batch
            if i % 10000 == 0 or i >= total:
                print(f"→ {min(i,total):,}/{total:,}")
    print("Ingesta completa ✅")

print("\n" + "="*70)
print("INGESTA COMPLETA DE COMPLAINTS")
print("="*70)
ingest(df, CYPHER_UPSERT_COMPLAINT, batch=1000)

with driver.session(database="neo4j") as s:
    complaints = s.run("MATCH (c:Complaint) RETURN count(c) AS n").single()['n']
    
    print(f"\n[OK] Ingesta completada!")
    print(f"   Complaints: {complaints:,}")


INGESTA COMPLETA DE COMPLAINTS
→ 10,000/512,725
→ 20,000/512,725
→ 30,000/512,725
→ 40,000/512,725
→ 50,000/512,725
→ 60,000/512,725
→ 70,000/512,725
→ 80,000/512,725
→ 90,000/512,725
→ 100,000/512,725
→ 110,000/512,725
→ 120,000/512,725
→ 130,000/512,725
→ 140,000/512,725
→ 150,000/512,725
→ 160,000/512,725
→ 170,000/512,725
→ 180,000/512,725
→ 190,000/512,725
→ 200,000/512,725
→ 210,000/512,725
→ 220,000/512,725
→ 230,000/512,725
→ 240,000/512,725
→ 250,000/512,725
→ 260,000/512,725
→ 270,000/512,725
→ 280,000/512,725
→ 290,000/512,725
→ 300,000/512,725
→ 310,000/512,725
→ 320,000/512,725
→ 330,000/512,725
→ 340,000/512,725
→ 350,000/512,725
→ 360,000/512,725
→ 370,000/512,725
→ 380,000/512,725
→ 390,000/512,725
→ 400,000/512,725
→ 410,000/512,725
→ 420,000/512,725
→ 430,000/512,725
→ 440,000/512,725
→ 450,000/512,725
→ 460,000/512,725
→ 470,000/512,725
→ 480,000/512,725
→ 490,000/512,725
→ 500,000/512,725
→ 510,000/512,725
→ 512,725/512,725
Ingesta completa ✅

[OK] Ingesta completa